In [1]:
import pandas as pd
import numpy as np
import os, sys
import itertools
import glob
import re
import difflib
from bisect import bisect_left, bisect_right


In [2]:
notebook_dir = os.path.dirname(os.getcwd())
source_data_path=os.path.join(notebook_dir, "Common Source Data")
sys.path.append(source_data_path)
from country_codes import countries
from country_regions import regions
from country_regions import region_iso3
from country_regions import iso3_region


In [3]:
#Adding in WAHIS country-wide reports of vaccination to reference
wahis_country_wide_poultry= pd.read_csv(os.path.join(source_data_path, 'WAHIS data','poultry_country-wide_vaccinated.csv'))
wahis_country_wide_cattle= pd.read_csv(os.path.join(source_data_path, 'WAHIS data','cattle_country-wide_vaccinated.csv'))
wahis_country_wide_pigs= pd.read_csv(os.path.join(source_data_path, 'WAHIS data','swine_country-wide_vaccinated.csv'))

wahis_country_wide_vaccination=pd.concat([wahis_country_wide_poultry,wahis_country_wide_cattle,wahis_country_wide_pigs])


col = 'Number of vaccinated'  # set the column used below
wahis_country_wide_vaccination[col] = (
    wahis_country_wide_vaccination[col]
    .astype(str).str.replace('-', '', regex=False)  # remove '-' if it's just a thousands/placeholder
    .replace({'': np.nan})                          # empty -> NaN
    .astype(float)
)

mask = wahis_country_wide_vaccination[col].notna() & wahis_country_wide_vaccination[col].astype(str).str.strip().ne('')
wahis_country_wide_vaccination['Source'] = np.where(
    mask,
    'WAHIS country level report',
    'WAHIS country level report but vaccinated count not provided'
)

wahis_country_wide_vaccination['ISO3']=[countries.get(country, 'Unknown code:'+country) for country in wahis_country_wide_vaccination['Country']]


priority = {
    'WAHIS country level report': 0,
    'WAHIS country level report but vaccinated count not provided': 1
}
df = (
    wahis_country_wide_vaccination
      .assign(_p=wahis_country_wide_vaccination['Source'].map(priority).fillna(99))
      .sort_values(['Year', 'ISO3', 'Disease', '_p'])
      .drop_duplicates(['Year','ISO3','Disease'], keep='first')
      .drop(columns='_p')
)
wahis_country_wide_vaccination['Vaccinate if vaccine exists and not prohibited']='Yes'


wahis_country_wide_vaccination.loc[wahis_country_wide_vaccination['Disease'] == 'Newcastle disease virus (Inf. with)', 'Disease']='Newcastle disease (velogenic)'

# Duplicate velogenic rows and relabel as "Newcastle disease", then append to main df (to reconcile with USA  non-velogenic vaccination as it is disease-free from velogenic variant)
mask = wahis_country_wide_vaccination['Disease'].eq('Newcastle disease (velogenic)')
to_dup = wahis_country_wide_vaccination.loc[mask].copy()
to_dup['Disease'] = 'Newcastle disease'

# Append the rows back into the main dataframe
wahis_country_wide_vaccination = pd.concat([wahis_country_wide_vaccination, to_dup], ignore_index=True)


wahis_country_wide_vaccination

,Year,Semester,Region,Country,Disease,Animal category,Species,Vaccine type,Number of vaccinated,Source,ISO3,Vaccinate if vaccine exists and not prohibited
0,2005,Jul-Dec 2005,Asia,Afghanistan,Newcastle disease (velogenic),Domestic,Birds,-,95574.0,WAHIS country level report,AFG,Yes
1,2005,Jul-Dec 2005,Europe,Albania,Newcastle disease (velogenic),Domestic,Birds,-,2046783.0,WAHIS country level report,ALB,Yes
2,2005,Jul-Dec 2005,Africa,Algeria,Avian infectious bronchitis,Domestic,Birds,-,NaN,WAHIS country level report but vaccinated coun...,DZA,Yes
3,2005,Jul-Dec 2005,Africa,Algeria,Fowl pox (-2005),Domestic,Birds,-,NaN,WAHIS country level report but vaccinated coun...,DZA,Yes
4,2005,Jul-Dec 2005,Africa,Algeria,Infectious bursal disease (Gumboro disease),Domestic,Birds,-,NaN,WAHIS country level report but vaccinated coun...,DZA,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...
36506,2024,Jul-Dec 2024,Asia,Uzbekistan,Newcastle disease,Domestic,Birds,-,NaN,WAHIS country level report but vaccinated coun...,UZB,Yes
36507,2025,Jan-Jun 2025,Asia,Bangladesh,Newcastle disease,Domestic,Birds,Live Attenuated Vaccine,118781802.0,WAHIS country level report,BGD,Yes
36508,2025,Jan-Jun 2025,Asia,Bangladesh,Newcastle disease,Domestic,Birds,-,NaN,WAHIS country level report but vaccinated coun...,BGD,Yes
36509,2025,Jan-Jun 2025,Europe,Estonia,Newcastle disease,Domestic,Birds,Live Attenuated Vaccine,165000.0,WAHIS country level report,EST,Yes


In [4]:
#Looking across all species here, and noting where there is "Official vaccination coverage", since this code is to filter whether imputations should be generated
    #for ANY animal in the country
#There is a final step of vaccination prohibition implementation in "Combine ALL vaccination coverage dataframes.ipynb" by species
official_vaccination_measure_cattle=pd.read_csv(os.path.join(source_data_path, 'WAHIS data','cattle_control_measures.csv'))
official_vaccination_measure_poultry=pd.read_csv(os.path.join(source_data_path, 'WAHIS data','poultry_control_measures.csv'))
official_vaccination_measure_pigs=pd.read_csv(os.path.join(source_data_path, 'WAHIS data','swine_control_measures.csv'))

official_vaccination_measure=pd.concat([official_vaccination_measure_cattle,official_vaccination_measure_poultry,
                                        official_vaccination_measure_pigs])

official_vaccination_measure['ISO3']=[countries.get(country, 'Unknown code:'+country) for country in official_vaccination_measure['Country']]

                                              
WAHIS_official_vaccination = (
    official_vaccination_measure
      .loc[official_vaccination_measure['Control measure'].eq('Official vaccination'),
           ['ISO3', 'Disease', 'Year']]
      .drop_duplicates()
      .reset_index(drop=True)
)

WAHIS_official_vaccination.loc[WAHIS_official_vaccination['Disease'] == 'Newcastle disease virus (Inf. with)', 'Disease']='Newcastle disease (velogenic)'
# Duplicate velogenic rows and relabel as "Newcastle disease", then append to main df

mask = WAHIS_official_vaccination['Disease'].eq('Newcastle disease (velogenic)')
to_dup = WAHIS_official_vaccination.loc[mask].copy()
to_dup['Disease'] = 'Newcastle disease'

# If 'Disease' is categorical, make sure the new label exists
if pd.api.types.is_categorical_dtype(WAHIS_official_vaccination['Disease']):
    WAHIS_official_vaccination['Disease'] = WAHIS_official_vaccination['Disease'].cat.add_categories(['Newcastle disease'])

# Append the duplicated rows back into the main dataframe
WAHIS_official_vaccination = pd.concat([WAHIS_official_vaccination, to_dup], ignore_index=True)

WAHIS_official_vaccination['Vaccinate if vaccine exists and not prohibited']='Yes'
WAHIS_official_vaccination['Source']="WAHIS 'official vaccination' control measure"

In [5]:
wahis_reports_poultry= pd.read_csv(os.path.join(source_data_path, 'WAHIS data','poultry_admin_div_reports.csv'))
wahis_reports_cattle= pd.read_csv(os.path.join(source_data_path, 'WAHIS data','cattle_admin_div_reports.csv'))
wahis_reports_pigs= pd.read_csv(os.path.join(source_data_path, 'WAHIS data','swine_admin_div_reports.csv'))

wahis_admin_vaccination=pd.concat([wahis_reports_poultry,wahis_reports_cattle,wahis_reports_pigs])

wahis_admin_vaccination['Vaccinated'] = wahis_admin_vaccination['Vaccinated'].replace('-',0).astype(float)
wahis_admin_vaccination=wahis_admin_vaccination[wahis_admin_vaccination['Vaccinated']>0]
wahis_admin_vaccination = wahis_admin_vaccination.drop_duplicates(subset=["Year", "Country", "Disease"]).reset_index(drop=True).loc[:,['Year','Country','Disease']]


wahis_admin_vaccination['ISO3']=[countries.get(country, 'Unknown code:'+country) for country in wahis_admin_vaccination['Country']]
wahis_admin_vaccination['Vaccinate if vaccine exists and not prohibited']='Yes'
wahis_admin_vaccination['Source']='WAHIS administrative division report'

In [6]:
combined_vaccinate_df=pd.concat([WAHIS_official_vaccination,wahis_admin_vaccination,wahis_country_wide_vaccination])

priority = {
    "WAHIS official control measure": 0,
    "WAHIS country level report": 1,
    "WAHIS administrative division report": 2,
    "WAHIS country level report but number vaccinated not provided": 3
}

combined_vaccinate_df = (
    combined_vaccinate_df
        .assign(_p=combined_vaccinate_df["Source"].map(priority).fillna(99))
        .sort_values(["ISO3", "Disease", "Year", "_p"])
        .drop_duplicates(subset=["ISO3", "Disease", "Year"], keep="first")
        .drop(columns="_p")
        .reset_index(drop=True)
)

combined_vaccinate_df=combined_vaccinate_df.drop(columns='Country')

In [7]:
disease_presence_or_absence=pd.read_csv(os.path.join(notebook_dir,'Common Source Data','Processed Data','Multisource','domestic_disease_presence_or_absence_filter.csv'))

In [8]:
reference_disease_iso3s_years=pd.read_csv(os.path.join(notebook_dir,'Imputation','df_impute_skeleton.csv')).drop(columns=['Unnamed: 0'])
diseases=[i for i in reference_disease_iso3s_years.columns if ('incidence' in i ) | ('vaccine' in i)]
poss_diseases=np.unique([i.split('_')[0] for i in diseases])
poss_ISO3s=np.unique(reference_disease_iso3s_years['ISO3']).tolist()+['FLK'] #FLK reports prohibition information, so may include as well for other diease absence info
poss_years=[int(i) for i in np.unique(reference_disease_iso3s_years['Year'])]


In [9]:
#Adding WAHIS 'disease situtation' reports
WAHIS_disease_sit=pd.read_csv(os.path.join(source_data_path, 'WAHIS data','Disease situation.csv'))
WAHIS_disease_sit.loc[WAHIS_disease_sit['Occurence code'] == 'Disease limited to one or more zones', 'Disease status'] = 'Present'
WAHIS_disease_sit.rename(columns={'Disease status':'Status'},inplace=True)

In [10]:
WAHIS_disease_sit['ISO3']=[countries.get(country, 'Unknown code:'+country) for country in WAHIS_disease_sit['Country']]

#Two cities that are listed as countries are dropped here
WAHIS_disease_sit = WAHIS_disease_sit[~WAHIS_disease_sit['Country'].isin(['Ceuta', 'Melilla'])]

In [11]:
#Fuzzy matching FAO/disease free report disease names to match WAHIS conventions
def clean_name(s: str) -> str:
    """Lowercase; remove parenthetical garbage & punctuation."""
    if s is None:
        return ""
    s = str(s).casefold()
    s = re.sub(r"\([^)]*\)", " ", s)        # drop parenthesis contents
    s = s.replace("-", " ")
    s = re.sub(r"[/_,.]", " ", s)
    s = re.sub(r"\b(virus|viruses|disease|infection|infections|inf|with)\b", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def best_match_one(target, choices_clean, choices_raw, scorer="auto"):
    """Return (best_raw, score) for target among choices_raw."""
    t = clean_name(target)
    if not t or not choices_raw:
        return None, 0.0

    else:
        # difflib fallback on cleaned names
        matches = difflib.get_close_matches(t, choices_clean, n=1, cutoff=0.0)
        if not matches:
            return None, 0.0
        best = matches[0]
        score = difflib.SequenceMatcher(a=t, b=best).ratio() * 100.0
        idx = choices_clean.index(best)
        return choices_raw[idx], score

def map_poss_to_df(
    df_new: pd.DataFrame,
    poss_diseases: list,
    threshold: float = 78.0,
    one_to_one: bool = False,
):
    """Map each poss_diseases item to its best Disease (or None if < threshold)."""
    fao_unique_raw = sorted(set(df_new["Disease"].astype(str)))
    fao_unique_clean = [clean_name(x) for x in fao_unique_raw]

    # Greedy matching (optionally one-to-one)
    used_idx = set()
    results = []
    for poss in poss_diseases:
        best_raw, score = best_match_one(poss, fao_unique_clean, fao_unique_raw)
        matched = None
        if best_raw is not None and score >= threshold:
            if one_to_one:
                # If already used, pick next best by temporarily removing it and retrying
                # (simple greedy uniqueness)
                tmp_clean = list(fao_unique_clean)
                tmp_raw = list(fao_unique_raw)
                while True:
                    idx = tmp_raw.index(best_raw)
                    if idx not in used_idx:
                        used_idx.add(idx)
                        matched = best_raw
                        break
                    # remove the used one and search again
                    tmp_clean.pop(idx); tmp_raw.pop(idx)
                    best_raw, score = best_match_one(poss, tmp_clean, tmp_raw)
                    if best_raw is None or score < threshold:
                        matched = None
                        break
            else:
                matched = best_raw

        results.append({
            "poss_disease": poss,
            "matched_fao_disease": matched,
            "score": round(score, 1)
        })

    mapping_df = pd.DataFrame(results)

    fao_to_poss = {
        row["matched_fao_disease"]: row["poss_disease"]
        for _, row in mapping_df.dropna(subset=["matched_fao_disease"]).iterrows()
    }
    return mapping_df, fao_to_poss

In [12]:
#Making skeleton dataframe for all diseases to reference
combinations = list(itertools.product(poss_ISO3s, poss_years, poss_diseases))

# Make dataframe
reconciled_present_absent_or_unknown_vaccination = pd.DataFrame(combinations, columns=["ISO3", "Year", "Disease"])
reconciled_present_absent_or_unknown_vaccination["Region"] = (
    reconciled_present_absent_or_unknown_vaccination["ISO3"].map(iso3_region)
)
reconciled_present_absent_or_unknown_vaccination['Vaccinate if vaccine exists and not prohibited']='Unknown'
reconciled_present_absent_or_unknown_vaccination['Source']=np.nan
reconciled_present_absent_or_unknown_vaccination['Year data'] = pd.Series([pd.NA] * len(reconciled_present_absent_or_unknown_vaccination), dtype='Int64')

reconciled_present_absent_or_unknown_vaccination['Source']=reconciled_present_absent_or_unknown_vaccination['Source'].astype("string")


In [13]:
disease_presence_or_absence[disease_presence_or_absence.duplicated(['ISO3','Disease','Year'])]

,ISO3,Disease,Year,Region,Any cases in country suspected (wildlife included),Source,Year data,Status in livestock


In [14]:
np.unique(disease_presence_or_absence['Status in livestock'])

array(['Absent', 'Present', 'Unknown'], dtype=object)

In [15]:
#Assigning here that if there is missing data for "Mycobacterium tuberculosis complex (Inf. with)(2019-)", next look at the other
    #diseases listed for any available information ("Mycobacterium tuberculosis complex (Inf. with)(2019-)" is parent category)
    #i.e., 'aliases'
    
DISEASE_PRIORITIES = {
    "Mycobacterium tuberculosis complex (Inf. with)(2019-)": [
        "Mycobacterium tuberculosis complex (Inf. with)(2019-)",
        "Bovine tuberculosis (-2018)",
        "Mycobacterium tuberculosis (Inf. with)(-2017)",
    ],
    "Bovine tuberculosis (-2018)": [
        "Bovine tuberculosis (-2018)",
    ],
    "Mycobacterium tuberculosis (Inf. with)(-2017)": [
        "Mycobacterium tuberculosis (Inf. with)(-2017)",
    ],

    # LPAI naming split (virtually a rename across years, as most LPAI serotypes are human-transmissible)
    "Low pathogenic avian influenza (poultry) (2006-2021)": [
        "Low pathogenic avian influenza (poultry) (2006-2021)",
        "Low pathogenicity avian influenza viruses transmissible to humans (Inf. with) (2022-)",
    ],
    "Low pathogenicity avian influenza viruses transmissible to humans (Inf. with) (2022-)": [
        "Low pathogenicity avian influenza viruses transmissible to humans (Inf. with) (2022-)",
        "Low pathogenic avian influenza (poultry) (2006-2021)",
    ]
    
    ,
}
def _prio_list(d):  # default: only itself
    return DISEASE_PRIORITIES.get(d, [d])

def update_vaccination_with_aliases(df, combined, presence):
    COL_V = "Vaccinate if vaccine exists and not prohibited"
    COL_S = "Status in livestock"

    df = df.copy()
    for t in (df, combined, presence):
        t["Year"] = pd.to_numeric(t["Year"], errors="coerce").astype("Int64")

    if "Year data" not in df.columns:
        df["Year data"] = pd.Series([pd.NA]*len(df), dtype="Int64")
    if "Source" not in df.columns:
        df["Source"] = ""

    # Making note of when Yes/No data is found
    df["decided"] = df[COL_V].isin(["Yes","No"]).fillna(False)

    comb_exact = {}
    for iso, dis, yr, src, val in combined[["ISO3","Disease","Year","Source",COL_V]].dropna(subset=["Year"]).itertuples(index=False, name=None):
        comb_exact[(iso, dis, yr)] = (val, src)

    # Dictionary to contain latest information available for each country-disease pair (sorted arrays for latest-prior)
    comb_by_pair = {}
    for (iso, dis), grp in combined.dropna(subset=["Year"]).groupby(["ISO3","Disease"]):
        g = grp.sort_values("Year")
        comb_by_pair[(iso, dis)] = (
            g["Year"].astype(int).to_list(),
            g[COL_V].tolist(),
            g["Source"].astype("string").to_list(),
        )

    pres_c = presence.copy()

    # Noting the presence, and source of each disase present report by ISO3, Disease, Year in respective dictionary
    pres_status, pres_source = {}, {}
    for iso, dis, yr, st, src in pres_c[["ISO3","Disease","Year",COL_S,"Source"]].itertuples(index=False, name=None):
        pres_status[(iso, dis, yr)] = st
        pres_source[(iso, dis, yr)] = "" if pd.isna(src) else str(src)

    pres_by_pair = {}
    pres_present_years = {}
    for (iso, dis), grp in pres_c.groupby(["ISO3","Disease"]):
        g = grp.sort_values("Year")
        years  = g["Year"].astype(int).to_list()
        status = g[COL_S].tolist()
        pres_by_pair[(iso, dis)] = (years, status)
        pres_present_years[(iso, dis)] = [y for y, s in zip(years, status) if s == "Present"]
    
    
    #Conditions: Inputs to provide as source per condition
    SRC_YES_PRESENT_PREFIX = "Inferred vaccination - disease presence"
    def _fmt_absent(src):
        return f"Inferred no vaccination - no past disease reports, current absence ({src})" if src else "Inferred no vaccination - no past disease reports, current absence"
    def _fmt_past(src):
        return f"Inferred vaccination - past disease presence ({src})" if src else "Inferred vaccination - past presence of disease"

    SRC_YES_MAJ = "Inferred vaccination - majority of countries in the same Global Health Data Exchange region vaccinate"
    SRC_NO_MAJ  = "Inferred no vaccination - majority of countries in the same Global Health Data Exchange region do not vaccinate"

    #Check if vaccination/nonvaccination status is diretly reported
    for (iso, dis), block in df.groupby(["ISO3","Disease"], sort=False):
        prio_dis = _prio_list(dis)
        for row in block.itertuples(index=True):
            idx = row.Index
            if bool(df.at[idx, "decided"]) or pd.isna(row.Year):
                continue
            y = int(row.Year)

            assigned = False
            for d_alias in prio_dis:
                hit = comb_exact.get((iso, d_alias, y))
                if hit is not None:
                    v, s = hit
                    df.at[idx, COL_V] = v
                    df.at[idx, "Source"] = s
                    df.at[idx, "Year data"] = y
                    df.at[idx, "decided"] = True
                    assigned = True
                    break
            if assigned: 
                continue

            for d_alias in prio_dis:
                pair = comb_by_pair.get((iso, d_alias))
                if not pair: 
                    continue
                years, vals, srcs = pair
                pos = bisect_left(years, y) - 1
                if pos >= 0:
                    df.at[idx, COL_V] = vals[pos]
                    df.at[idx, "Source"] = srcs[pos]
                    df.at[idx, "Year data"] = years[pos]
                    df.at[idx, "decided"] = True
                    assigned = True
                    break
            if assigned: 
                continue

            # 3) Fallback to disease presence - (per-alias; stop at first assignment)
            for d_alias in prio_dis:
                st_now = pres_status.get((iso, d_alias, y))

                # 3b: Present now -> Yes (with exact source)
                if st_now == "Present":
                    src_now = pres_source.get((iso, d_alias, y), "")
                    df.at[idx, COL_V] = "Yes"
                    df.at[idx, "Source"] = SRC_YES_PRESENT_PREFIX + ' ('+src_now+')'
                    df.at[idx, "Year data"] = y
                    df.at[idx, "decided"] = True
                    assigned = True
                    break

                # 3a: Absent now AND all recorded years -> No
                if st_now == "Absent":
                    pair = pres_by_pair.get((iso, d_alias))
                    if pair:
                        years, status = pair
                        pos = bisect_right(years, y) - 1
                        if pos >= 0 and all(s == "Absent" for s in status[:pos+1]):
                            src_now = pres_source.get((iso, d_alias, y), "")
                            df.at[idx, COL_V] = "No"
                            df.at[idx, "Source"] = _fmt_absent(src_now)  # exact absence source
                            df.at[idx, "Year data"] = y
                            df.at[idx, "decided"] = True
                            assigned = True
                            break
            if assigned: 
                continue

            # 4) Check for past presence (before current year) (use latest < y, with exact source)
            for d_alias in prio_dis:
                pyears = pres_present_years.get((iso, d_alias), [])
                if not pyears:
                    continue
                pos = bisect_right(pyears, y) - 1
                if pos >= 0:
                    y_match = pyears[pos]
                    src_match = pres_source.get((iso, d_alias, y_match), "")
                    df.at[idx, COL_V] = "Yes"
                    df.at[idx, "Source"] = _fmt_past(src_match)  # exact past presence source
                    df.at[idx, "Year data"] = y_match
                    df.at[idx, "decided"] = True
                    assigned = True
                    break
            # if not assigned, remains Unknown for now

    
    #Step 5: Regional majority voting
    undecided = (~df["decided"]) & (df[COL_V].isna() | (df[COL_V] == "Unknown"))

    # pre-count Yes/No per (Disease,Year,Region)
    counts = {}
    base = df.loc[df[COL_V].isin(["Yes","No"]), ["Disease","Year","Region",COL_V]]
    if not base.empty:
        for (d, y, r, v), n in base.value_counts().items():
            if pd.isna(y): 
                continue
            key = (d, int(y), r)
            if key not in counts:
                counts[key] = {"Yes":0, "No":0}
            counts[key][v] += n

    for row in df.loc[undecided, ["Disease","Year","Region"]].itertuples(index=True, name=None):
        idx, d, y, r = row
        if pd.isna(y):
            continue
        y = int(y)
        prio_dis = _prio_list(d)

        yes = no = 0
        for dd in prio_dis:
            c = counts.get((dd, y, r))
            if c:
                yes += c.get("Yes", 0)
                no  += c.get("No", 0)

        if yes > no:
            df.at[idx, COL_V] = "Yes"
            df.at[idx, "Source"] = SRC_YES_MAJ
            df.at[idx, "Year data"] = y
            df.at[idx, "decided"] = True
        elif no > yes:
            df.at[idx, COL_V] = "No"
            df.at[idx, "Source"] = SRC_NO_MAJ
            df.at[idx, "Year data"] = y
            df.at[idx, "decided"] = True
        
        # ties: leave as Unknown/undecided

    return df


In [16]:
reconciled_updated_vaccination_status = update_vaccination_with_aliases(
    reconciled_present_absent_or_unknown_vaccination,
    combined_vaccinate_df,
    disease_presence_or_absence
)

In [17]:
reconciled_updated_vaccination_status.drop(columns=['decided']).to_csv(os.path.join(notebook_dir,'Common Source Data','Processed data','Multisource','vaccination_filter.csv'),
                         index=False)